# Khojki OCR — bounded pilot
Prepared offline before connecting. Upload the paired ZIP when prompted. One 100-step experiment; test set excluded. Starting balance 168.49 units; minimum reserve 75; pilot budget <=5 units. Download results and terminate the runtime after this batch. Existing notebooks are untouched.


In [ ]:
# Connect only after this notebook is uploaded. Select T4 GPU.
import sys, os, json, time, hashlib, zipfile, subprocess
from pathlib import Path
from google.colab import files
START = time.monotonic()
BUNDLE = Path('/content/khojki-cloud-pilot.zip')
if not BUNDLE.exists():
    uploaded = files.upload()
    del uploaded
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == 'de873349552b951435f044af6460baea6abd59f5d6a53acedac244864af741cd'
ROOT = Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z: z.extractall('/content')
for name, expected in json.loads((ROOT/'cloud-manifest.json').read_text()).items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
subprocess.run([sys.executable,'-m','pip','install','-q','freetype-py','uharfbuzz','scipy','opencv-python-headless'],check=True,timeout=180)
os.environ['PYTHONPATH'] = str(ROOT/'src')
import torch
assert torch.cuda.is_available(), 'Stop: GPU not connected.'
print('GPU:', torch.cuda.get_device_name(0))
assert all(r['split'] != 'test' for r in json.loads((ROOT/'data/annotations.json').read_text()))
print('Verified 21 train/dev lines. No test data present.')


In [ ]:
# Fixed pilot: one 100-step warm start, no automatic sweep or long run.
OUT = ROOT/'experiments/cloud-pilot-100'
def command(args, log, timeout):
    with open(log,'w') as f:
        p = subprocess.run([sys.executable,*map(str,args)], stdout=f, stderr=subprocess.STDOUT, timeout=timeout)
    print(Path(log).read_text()[-10000:])
    p.check_returncode()
smoke = ROOT/'experiments/cloud-smoke-dev.json'
if not smoke.exists():
    command([ROOT/'scripts/evaluate.py','--checkpoint',ROOT/'experiments/curve-10/best.pt','--split','dev','--device','cuda','--output',smoke],ROOT/'experiments/cloud-smoke.log',120)
baseline = json.loads(smoke.read_text())['summary']
print('BASELINE', baseline)
assert baseline['errors'] == 10 and baseline['characters'] == 155, 'Unexpected smoke result: investigate before training.'
if not (OUT/'history.json').exists():
    command([ROOT/'scripts/finetune.py','--checkpoint',ROOT/'experiments/curve-10/best.pt','--steps','100','--device','cuda','--lr','0.00005','--seed','71','--output','cloud-pilot-100'],ROOT/'experiments/cloud-pilot.log',600)
else:
    print('Saved pilot exists; skipping training. Use a new named experiment for new settings.')
history = json.loads((OUT/'history.json').read_text())
best = min(history,key=lambda r:r['validation_cer'])
print('PILOT', {k:v for k,v in best.items() if k!='lines'})
print('SELECT', 'pilot' if best['errors'] < baseline['errors'] else 'original curve-10')
print('Cloud session seconds:', time.monotonic()-START)


In [ ]:
# Save model, optimizer, RNG, metrics and logs BEFORE disconnecting.
RESULT = Path('/content/khojki-pilot-results.zip')
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in (ROOT/'experiments').rglob('*'):
        if p.is_file() and ('cloud-' in str(p.relative_to(ROOT))):
            z.write(p,str(p.relative_to(ROOT)))
print('RESULT', RESULT.name, RESULT.stat().st_size, hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))
# After verifying the download, use Runtime > Disconnect and delete runtime.
